# nnUNet Baseline — BraTS 2024 MEN-RT (Meningioma GTV Segmentation)

**Smoke test:** 50 training cases · 5-fold CV · 50 epochs per fold

**Before running:**
1. Attach BraTS-MEN-RT dataset (right panel → Add Data)
2. Enable Internet (right panel → Internet → ON)
3. Enable GPU T4 (right panel → Accelerator → GPU T4 x2)
4. Run cells top to bottom

**To resume a fold after disconnect:** re-run the cell with `--continue` added.

**Pipeline:**
```
GPU check → Install → Setup → Verify dataset → Prepare (50 cases)
→ Splits → Install trainer → Plan & preprocess
→ Train Fold 0 → Train Folds 1-4
→ Metrics table → Plots → Training curves
```

In [ ]:
# Cell 1: GPU check
!nvidia-smi

In [ ]:
# Cell 2: Install dependencies
!pip install -q nnunet==1.7.1
!pip install -q pyyaml simpleitk scipy pandas matplotlib
print('Installation complete.')

In [ ]:
# Cell 3: Setup — paths, constants, environment variables
# Re-run this cell if the kernel restarts before training completes.
import os, sys

TRAIN_ROOT   = '/kaggle/input/datasets/maryampervaiz24029/brats-men-rt/BraTS2024-MEN-RT-TrainingData/BraTS-MEN-RT-Train-v2'
VAL_ROOT     = '/kaggle/input/datasets/maryampervaiz24029/brats-men-rt/BraTS2024-MEN-RT-ValidationData/BraTS-MEN-RT-Val-v1'

TASK_ID      = 502
TASK_NAME    = f'Task{TASK_ID}_BraTSMENRT'
TRAINER_NAME = 'nnUNetTrainerV2_MENRT'
PLANS        = 'nnUNetPlansv2.1'
NETWORK      = '3d_fullres'

RAW_BASE     = '/kaggle/working/nnUNet_raw_data_base'
PREPROCESSED = '/kaggle/working/nnUNet_preprocessed'
RESULTS      = '/kaggle/working/nnUNet_results'
ARTIFACTS    = '/kaggle/working/menrt_artifacts'

TRAIN_LIMIT  = 50
VAL_LIMIT    = 20
N_FOLDS      = 5
N_BINS       = 5
SEED         = 42
MAX_EPOCHS   = 50

os.environ['nnUNet_raw_data_base'] = RAW_BASE
os.environ['nnUNet_preprocessed']  = PREPROCESSED
os.environ['RESULTS_FOLDER']       = RESULTS
os.environ['NNUNET_MAX_EPOCHS']    = str(MAX_EPOCHS)
os.environ['PYTORCH_ALLOC_CONF']   = 'expandable_segments:True'

for d in [RAW_BASE, PREPROCESSED, RESULTS, ARTIFACTS]:
    os.makedirs(d, exist_ok=True)

print(f'Task      : {TASK_NAME}')
print(f'Trainer   : {TRAINER_NAME}')
print(f'Train cases: {TRAIN_LIMIT}  (stratified by GTV volume)')
print(f'Folds     : {N_FOLDS} x {MAX_EPOCHS} epochs')
print(f'Results   : {RESULTS}')

In [ ]:
# Cell 4: Verify dataset accessibility
train_ok = os.path.exists(TRAIN_ROOT)
val_ok   = os.path.exists(VAL_ROOT)

if train_ok:
    n_tr = len([d for d in os.listdir(TRAIN_ROOT) if os.path.isdir(os.path.join(TRAIN_ROOT, d))])
    print(f'Train root : OK  ({n_tr} cases)  {TRAIN_ROOT}')
else:
    print(f'Train root : NOT FOUND  {TRAIN_ROOT}')

if val_ok:
    n_val = len([d for d in os.listdir(VAL_ROOT) if os.path.isdir(os.path.join(VAL_ROOT, d))])
    print(f'Val root   : OK  ({n_val} cases)  {VAL_ROOT}')
else:
    print(f'Val root   : NOT FOUND  {VAL_ROOT}')

if not train_ok or not val_ok:
    raise FileNotFoundError(
        'Dataset not found. Attach BraTS-MEN-RT in Kaggle → Add Data.\n'
        'Run: !find /kaggle/input -maxdepth 4 -type d   to find correct path.'
    )
print('\nDataset verified.')

In [ ]:
# Cell 5: Prepare dataset — stratified 50 training cases + nnUNet folder structure
import json, shutil, pickle
import numpy as np
import SimpleITK as sitk

TASK_DIR  = f'{RAW_BASE}/nnUNet_raw_data/{TASK_NAME}'
IMAGES_TR = f'{TASK_DIR}/imagesTr'
LABELS_TR = f'{TASK_DIR}/labelsTr'
IMAGES_TS = f'{TASK_DIR}/imagesTs'
for d in [IMAGES_TR, LABELS_TR, IMAGES_TS]:
    os.makedirs(d, exist_ok=True)

# -- Compute GTV volume (mm3) for each training case -----------------------
print('Computing GTV volumes for stratified sampling...')
all_cases = sorted([
    d for d in os.listdir(TRAIN_ROOT)
    if os.path.isdir(os.path.join(TRAIN_ROOT, d))
])

case_vol = {}
for case in all_cases:
    gtv = os.path.join(TRAIN_ROOT, case, f'{case}_gtv.nii.gz')
    if os.path.exists(gtv):
        img = sitk.ReadImage(gtv)
        arr = sitk.GetArrayFromImage(img)
        sp  = img.GetSpacing()
        case_vol[case] = float(arr.sum()) * sp[0] * sp[1] * sp[2]

valid_cases = [c for c in all_cases if c in case_vol]
vols        = np.array([case_vol[c] for c in valid_cases])
print(f'Found {len(valid_cases)} cases with GTV | Volume range: {vols.min()/1000:.1f} - {vols.max()/1000:.1f} cm3')

# -- Stratified sampling ---------------------------------------------------
bin_edges = np.percentile(vols, np.linspace(0, 100, N_BINS + 1))
bin_edges[-1] += 1
bin_ids   = np.digitize(vols, bin_edges[1:])

rng      = np.random.default_rng(SEED)
selected = []
per_bin  = TRAIN_LIMIT // N_BINS
extra    = TRAIN_LIMIT % N_BINS

for b in range(N_BINS):
    pool   = [valid_cases[i] for i in range(len(valid_cases)) if bin_ids[i] == b]
    n      = per_bin + (1 if b < extra else 0)
    chosen = rng.choice(pool, min(n, len(pool)), replace=False).tolist()
    selected.extend(chosen)

print(f'Selected {len(selected)} training cases (stratified by GTV volume)')

# -- Copy training files ---------------------------------------------------
print('Copying training files...')
training_json = []
for case in selected:
    shutil.copy2(
        os.path.join(TRAIN_ROOT, case, f'{case}_t1c.nii.gz'),
        os.path.join(IMAGES_TR, f'{case}_0000.nii.gz')
    )
    shutil.copy2(
        os.path.join(TRAIN_ROOT, case, f'{case}_gtv.nii.gz'),
        os.path.join(LABELS_TR, f'{case}.nii.gz')
    )
    training_json.append({
        'image': f'./imagesTr/{case}_0000.nii.gz',
        'label': f'./labelsTr/{case}.nii.gz'
    })

# -- Copy val images (no labels available) ---------------------------------
all_val = sorted([d for d in os.listdir(VAL_ROOT) if os.path.isdir(os.path.join(VAL_ROOT, d))])
sel_val = rng.choice(all_val, min(VAL_LIMIT, len(all_val)), replace=False).tolist()
test_json = []
for case in sel_val:
    shutil.copy2(
        os.path.join(VAL_ROOT, case, f'{case}_t1c.nii.gz'),
        os.path.join(IMAGES_TS, f'{case}_0000.nii.gz')
    )
    test_json.append(f'./imagesTs/{case}_0000.nii.gz')

# -- Write dataset.json ----------------------------------------------------
dataset_json = {
    'name': TASK_NAME,
    'description': 'BraTS 2024 MEN-RT meningioma GTV segmentation (50-case smoke test)',
    'tensorImageSize': '4D',
    'reference': 'BraTS 2024 Challenge',
    'licence': 'see BraTS 2024',
    'release': '1.0',
    'modality': {'0': 'MRI_T1c'},
    'labels': {'0': 'background', '1': 'GTV'},
    'numTraining': len(selected),
    'numTest': len(sel_val),
    'training': training_json,
    'test': test_json,
}
with open(os.path.join(TASK_DIR, 'dataset.json'), 'w') as f:
    json.dump(dataset_json, f, indent=2)

# -- Save manifest ---------------------------------------------------------
manifest = {
    'selected_train_cases': selected,
    'selected_val_cases': sel_val,
    'strategy': 'stratified_gtv_volume_mm3',
    'n_bins': N_BINS,
    'seed': SEED,
}
with open(os.path.join(ARTIFACTS, 'subset_manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Training : {len(selected)} cases -> {IMAGES_TR}')
print(f'Test     : {len(sel_val)} cases -> {IMAGES_TS}')
print('dataset.json written.')

In [ ]:
# Cell 6: Create stratified 5-fold cross-validation splits
# Saved to splits_final.pkl — nnUNet reads this during training.
splits_dir = os.path.join(PREPROCESSED, TASK_NAME)
os.makedirs(splits_dir, exist_ok=True)

sel_vols   = np.array([case_vol[c] for c in selected])
bin_edges2 = np.percentile(sel_vols, np.linspace(0, 100, N_BINS + 1))
bin_edges2[-1] += 1
bin_ids2   = np.digitize(sel_vols, bin_edges2[1:])

rng2    = np.random.default_rng(SEED)
ordered = []
for b in range(N_BINS):
    bc = [selected[i] for i in range(len(selected)) if bin_ids2[i] == b]
    rng2.shuffle(bc)
    ordered.extend(bc)

splits = []
for fold in range(N_FOLDS):
    val_ids   = ordered[fold::N_FOLDS]
    train_ids = [c for c in ordered if c not in val_ids]
    splits.append({'train': train_ids, 'val': val_ids})
    print(f'Fold {fold}: {len(train_ids)} train / {len(val_ids)} val')

with open(os.path.join(splits_dir, 'splits_final.pkl'), 'wb') as f:
    pickle.dump(splits, f)
print('splits_final.pkl saved.')

In [ ]:
# Cell 7: Install custom trainer (nnUNetTrainerV2_MENRT)
# Thin subclass of nnUNetTrainerV2 that reads max epochs from NNUNET_MAX_EPOCHS env var.
import nnunet

TRAINER_CODE = '''\
import os
from nnunet.training.network_training.nnUNetTrainerV2 import nnUNetTrainerV2


class nnUNetTrainerV2_MENRT(nnUNetTrainerV2):
    """nnUNetTrainerV2 with epoch count from NNUNET_MAX_EPOCHS env var."""

    def __init__(self, plans_file, fold, output_folder=None, dataset_directory=None,
                 batch_dice=True, stage=None, unpack_data=True, deterministic=True, fp16=False):
        super().__init__(plans_file, fold, output_folder, dataset_directory,
                         batch_dice, stage, unpack_data, deterministic, fp16)
        self.max_num_epochs = int(os.environ.get("NNUNET_MAX_EPOCHS", "50"))
        self.save_every = 50  # checkpoint every 50 epochs
        print(f"[nnUNetTrainerV2_MENRT] max_num_epochs = {self.max_num_epochs}")
'''

nnunet_dir   = os.path.dirname(nnunet.__file__)
trainer_dir  = os.path.join(nnunet_dir, 'training', 'network_training')
trainer_file = os.path.join(trainer_dir, 'nnUNetTrainerV2_MENRT.py')

with open(trainer_file, 'w') as f:
    f.write(TRAINER_CODE)

# Verify import
from nnunet.training.network_training.nnUNetTrainerV2_MENRT import nnUNetTrainerV2_MENRT
print(f'Custom trainer installed and verified: {trainer_file}')

In [ ]:
# Cell 8: Plan and preprocess
# nnUNet automatically configures patch size, batch size, normalisation.
# Do NOT override these values — this is nnUNet's core strength.
!nnUNet_plan_and_preprocess -t {TASK_ID} --verify_dataset_integrity

## Training

nnUNet saves three checkpoints per fold automatically:
- `model_best.model` — highest validation Dice so far
- `model_latest.model` — saved every 50 epochs
- `model_final_checkpoint.model` — end of training

**If Kaggle disconnects:** re-run Cell 3 (restore env vars), then re-run the interrupted fold cell with `--continue` added to the command.

In [ ]:
# Cell 9: Train Fold 0 — 50 epochs
# Primary fold. Evaluate this fold first before running Folds 1-4.
# To resume if interrupted: add --continue at the end.
!nnUNet_train {NETWORK} {TRAINER_NAME} {TASK_NAME} 0

In [ ]:
# Cell 10: Train Fold 1 — 50 epochs
# To resume: add --continue
!nnUNet_train {NETWORK} {TRAINER_NAME} {TASK_NAME} 1

In [ ]:
# Cell 11: Train Fold 2 — 50 epochs
!nnUNet_train {NETWORK} {TRAINER_NAME} {TASK_NAME} 2

In [ ]:
# Cell 12: Train Fold 3 — 50 epochs
!nnUNet_train {NETWORK} {TRAINER_NAME} {TASK_NAME} 3

In [ ]:
# Cell 13: Train Fold 4 — 50 epochs
!nnUNet_train {NETWORK} {TRAINER_NAME} {TASK_NAME} 4

## Evaluation

nnUNet automatically evaluates each fold on its held-out validation cases and saves `summary.json`.

Metrics reported:
- **Dice (DSC)** — primary metric, overlap between prediction and ground truth (0–1, higher better)
- **HD95** — 95th percentile Hausdorff Distance in mm (lower better), measures boundary accuracy
- **Sensitivity** — fraction of true tumour voxels correctly predicted
- **Precision** — fraction of predicted tumour voxels that are correct

Run the cells below after **at least Fold 0** completes.

In [ ]:
# Cell 14: Parse metrics from all completed folds
import json
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

FOLD_DIR_BASE = os.path.join(RESULTS, 'nnUNet', NETWORK, TASK_NAME, f'{TRAINER_NAME}__{PLANS}')

fold_rows = []
case_rows = []

for fold in range(N_FOLDS):
    summary_path = os.path.join(FOLD_DIR_BASE, f'fold_{fold}', 'validation_raw', 'summary.json')
    if not os.path.exists(summary_path):
        print(f'Fold {fold}: not yet complete (summary.json missing)')
        continue

    with open(summary_path) as f:
        s = json.load(f)

    mean = s['results']['mean']['1']
    fold_rows.append({
        'Fold':      fold,
        'Dice':      round(mean.get('Dice', float('nan')), 4),
        'HD95 (mm)': round(mean.get('Hausdorff Distance 95', float('nan')), 2),
        'Sensitivity': round(mean.get('Recall', float('nan')), 4),
        'Precision':   round(mean.get('Precision', float('nan')), 4),
    })

    for c in s['results']['all']:
        case_id = os.path.basename(c.get('reference', '')).replace('.nii.gz', '')
        case_rows.append({
            'fold':        fold,
            'case_id':     case_id,
            'dice':        c['1'].get('Dice', float('nan')),
            'hd95':        c['1'].get('Hausdorff Distance 95', float('nan')),
            'sensitivity': c['1'].get('Recall', float('nan')),
            'precision':   c['1'].get('Precision', float('nan')),
        })

if not fold_rows:
    print('No completed folds found. Train at least Fold 0 first.')
else:
    df_folds = pd.DataFrame(fold_rows).set_index('Fold')
    df_cases = pd.DataFrame(case_rows)

    print('=' * 55)
    print('  nnUNet Baseline — Per-Fold Metrics')
    print('=' * 55)
    print(df_folds.to_string())
    print('-' * 55)

    completed = df_folds.index.tolist()
    print(f"\nMean Dice        : {df_folds['Dice'].mean():.4f} +/- {df_folds['Dice'].std():.4f}")
    print(f"Mean HD95        : {df_folds['HD95 (mm)'].mean():.2f} +/- {df_folds['HD95 (mm)'].std():.2f} mm")
    print(f"Mean Sensitivity : {df_folds['Sensitivity'].mean():.4f}")
    print(f"Mean Precision   : {df_folds['Precision'].mean():.4f}")
    print(f"\nCompleted folds  : {completed}")

    # Save summary CSV
    df_folds.to_csv(os.path.join(ARTIFACTS, 'fold_metrics.csv'))
    df_cases.to_csv(os.path.join(ARTIFACTS, 'per_case_metrics.csv'), index=False)
    print(f"\nCSVs saved to {ARTIFACTS}")

In [ ]:
# Cell 15: Publication-ready metric plots
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if 'df_folds' not in dir() or df_folds.empty:
    print('No metrics to plot. Run Cell 14 first.')
else:
    COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
    folds  = df_folds.index.tolist()
    labels = [f'Fold {f}' for f in folds]
    dices  = df_folds['Dice'].values
    hd95s  = df_folds['HD95 (mm)'].values

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # ── Plot 1: Mean Dice per fold (bar chart) ────────────────────────────
    ax = axes[0]
    bars = ax.bar(labels, dices, color=[COLORS[f % 5] for f in folds],
                  edgecolor='black', linewidth=0.8, width=0.55)
    ax.axhline(dices.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean = {dices.mean():.3f}')
    for bar, val in zip(bars, dices):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Mean Dice Score', fontsize=12)
    ax.set_title('Dice Score per Fold', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)

    # ── Plot 2: Per-case Dice boxplot ─────────────────────────────────────
    ax = axes[1]
    data_dice = [df_cases[df_cases['fold'] == f]['dice'].dropna().values for f in folds]
    bp = ax.boxplot(data_dice, patch_artist=True, labels=labels, widths=0.5)
    for patch, f in zip(bp['boxes'], folds):
        patch.set_facecolor(COLORS[f % 5])
        patch.set_alpha(0.7)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Dice Score', fontsize=12)
    ax.set_title('Dice Distribution per Fold', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    # ── Plot 3: Per-case HD95 boxplot ─────────────────────────────────────
    ax = axes[2]
    data_hd95 = [df_cases[df_cases['fold'] == f]['hd95'].dropna().values for f in folds]
    if any(len(h) > 0 for h in data_hd95):
        bp2 = ax.boxplot(data_hd95, patch_artist=True, labels=labels, widths=0.5)
        for patch, f in zip(bp2['boxes'], folds):
            patch.set_facecolor(COLORS[f % 5])
            patch.set_alpha(0.7)
        ax.set_ylabel('HD95 (mm)', fontsize=12)
        ax.set_title('HD95 Distribution per Fold', fontsize=13, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle(
        f'nnUNet Baseline — BraTS 2024 MEN-RT ({TRAIN_LIMIT} cases, {MAX_EPOCHS} epochs/fold)',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()

    fig_path = os.path.join(ARTIFACTS, 'nnunet_metrics.png')
    plt.savefig(fig_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {fig_path}')

    # ── Best / worst cases by Dice ────────────────────────────────────────
    df_sorted = df_cases.sort_values('dice', ascending=False)
    print('\n--- Top 5 cases (highest Dice) ---')
    print(df_sorted.head(5)[['case_id', 'fold', 'dice', 'hd95']].to_string(index=False))
    print('\n--- Bottom 5 cases (lowest Dice) ---')
    print(df_sorted.tail(5)[['case_id', 'fold', 'dice', 'hd95']].to_string(index=False))
    df_sorted.head(5).to_csv(os.path.join(ARTIFACTS, 'best_cases.csv'), index=False)
    df_sorted.tail(5).to_csv(os.path.join(ARTIFACTS, 'worst_cases.csv'), index=False)

In [ ]:
# Cell 16: Display training curves (loss + validation Dice per epoch)
from IPython.display import Image, display

for fold in range(N_FOLDS):
    progress_png = os.path.join(FOLD_DIR_BASE, f'fold_{fold}', 'progress.png')
    if os.path.exists(progress_png):
        print(f'\nFold {fold} training curves:')
        display(Image(filename=progress_png, width=700))
    else:
        print(f'Fold {fold}: progress.png not found (fold not complete yet)')

In [ ]:
# Cell 17: List all saved artifacts and checkpoints
print('=== Checkpoints ===')
for fold in range(N_FOLDS):
    fold_dir = os.path.join(FOLD_DIR_BASE, f'fold_{fold}')
    for ckpt in ['model_best.model', 'model_latest.model', 'model_final_checkpoint.model']:
        p = os.path.join(fold_dir, ckpt)
        status = 'OK' if os.path.exists(p) else 'missing'
        print(f'  fold_{fold}/{ckpt}: {status}')

print('\n=== Artifacts ===')
if os.path.exists(ARTIFACTS):
    for fname in sorted(os.listdir(ARTIFACTS)):
        fpath = os.path.join(ARTIFACTS, fname)
        size  = os.path.getsize(fpath)
        print(f'  {fname}  ({size/1024:.1f} KB)')

print(f'\nAll outputs under: /kaggle/working/')